In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import regex as re # For String searches
import plotly.graph_objects as go
import plotly.express as px
import time
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
### cell 0 ###

data = pd.read_csv('/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/fixed_year_2021_quarter_03.csv')

Mobile_df = pd.DataFrame([],columns=data.columns)
Broadband_df = pd.DataFrame([],columns=data.columns)

def col_name_corrections(df,correction_pair):
    if set(df.columns).intersection(set(correction_pair.keys())):
        df.rename(columns=correction_pair,inplace=True)
    return df

for dirname, _, filenames in os.walk('/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input'):
    for filename in filenames:
        meta_info = filename.split('/')[-1]
        data = pd.read_csv(dirname+'/'+filename,thousands=r',').convert_dtypes()
        data = col_name_corrections(data,{'Number of Record':'Number of Records'})
        data['Year'] = np.int64(re.search('year_(.*)_quarter',meta_info).group(1))
        data['Quarter'] = np.int64(re.search('quarter_(.*).csv',meta_info).group(1))
        if 'mobile' in meta_info:
            Mobile_df = pd.concat([Mobile_df,data])
        else:
            Broadband_df = pd.concat([Broadband_df,data]) 
print(Broadband_df.shape)
print(Mobile_df.shape)
Mobile_df = Mobile_df.astype({'Year':np.int64,'Quarter':np.int64})
Broadband_df = Broadband_df.astype({'Year':np.int64,'Quarter':np.int64})
Mobile_df.sort_values(by=['Year','Quarter'],ascending=[True,True],inplace=True)
Broadband_df.sort_values(by=['Year','Quarter'],ascending=[True,True],inplace=True)

factor = 1000

# build a GPU-side index that repeats each row index `factor` times
i = pd.Series(range(len(Mobile_df))).repeat(factor * 2)
# use `.take` to duplicate rows in one GPU call and reset the index
Mobile_df = Mobile_df.take(i).reset_index(drop=True)
Mobile_df.info()

Broadband_df = pd.concat([Broadband_df]*factor, ignore_index=True)
Broadband_df.info()

In [ ]:
### cell 1 ###

unique_countries_broadband = Broadband_df.groupby('Name').count()
unique_countries_broadband.head()

In [ ]:
### cell 2 ###

unique_countries_mobile = Mobile_df.groupby('Name').count()
unique_countries_mobile.head()

In [ ]:
### cell 3 ###

# Check for missing values
Mobile_df.isna().any()

In [ ]:
### cell 4 ###

Broadband_df.isna().any()

In [ ]:
### cell 5 ###

Mobile_df.shape

In [ ]:
### cell 6 ###

unique_countries_mobile[unique_countries_mobile.Year < 10]['Year']

In [ ]:
### cell 7 ###

unique_countries_broadband.loc[unique_countries_broadband.Year < 10, 'Year']

In [ ]:
### cell 8 ###

# Compute first and last per Name on GPU for Mobile
mobile_grp   = Mobile_df.groupby('Name')
mobile_first = mobile_grp.first()
mobile_last  = mobile_grp.last()

Mobile_Stats = mobile_last[[
    'Avg. Avg D Kbps',
    'Avg. Avg U Kbps',
    'Avg Lat Ms'
]].copy()
Mobile_Stats['Change_Download'] = (
    mobile_last['Avg. Avg D Kbps'] - mobile_first['Avg. Avg D Kbps']
)
Mobile_Stats['Change_Upload'] = (
    mobile_last['Avg. Avg U Kbps'] - mobile_first['Avg. Avg U Kbps']
)
Mobile_Stats['Change_Latency'] = (
    mobile_last['Avg Lat Ms'] - mobile_first['Avg Lat Ms']
)
Mobile_Stats = Mobile_Stats[[
    'Change_Download',
    'Change_Upload',
    'Change_Latency'
]]

# Compute first and last per Name on GPU for Broadband
broad_grp   = Broadband_df.groupby('Name')
broad_first = broad_grp.first()
broad_last  = broad_grp.last()

Broadband_Stats = broad_last[[
    'Avg. Avg D Kbps',
    'Avg. Avg U Kbps',
    'Avg Lat Ms'
]].copy()
Broadband_Stats['Change_Download'] = (
    broad_last['Avg. Avg D Kbps'] - broad_first['Avg. Avg D Kbps']
)
Broadband_Stats['Change_Upload'] = (
    broad_last['Avg. Avg U Kbps'] - broad_first['Avg. Avg U Kbps']
)
Broadband_Stats['Change_Latency'] = (
    broad_last['Avg Lat Ms'] - broad_first['Avg Lat Ms']
)
Broadband_Stats = Broadband_Stats[[
    'Change_Download',
    'Change_Upload',
    'Change_Latency'
]]

# Combine Mobile and Broadband Download changes on GPU
Total_Stats = pd.concat([
    Mobile_Stats['Change_Download'].rename('Mobile'),
    Broadband_Stats['Change_Download'].rename('Broadband')
], axis=1)

In [ ]:
### cell 9 ###

# Filter broadband stats for improved download speeds (0 < Change_Download < 3000)
ImprovedCountries_B = Broadband_Stats.loc[
    Broadband_Stats['Change_Download'].between(1, 2999)
]

In [ ]:
### cell 10 ###

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] < 8000) &
                                (Broadband_Stats['Change_Download'] > 3000)]

In [ ]:
### cell 11 ###

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] < 16000) &
                                (Broadband_Stats['Change_Download'] > 8000)]

In [ ]:
### cell 12 ###

# Select rows where Change_Download is strictly between 16 000 and 60 000 using cudf-compatible "inclusive" argument
ImprovedCountries_B = Broadband_Stats[
    Broadband_Stats['Change_Download']
                 .between(16000, 60000, inclusive='neither')
]

In [ ]:
### cell 13 ###

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] >= 60000)]

In [ ]:
### cell 14 ###

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] < 0)]
Countries = ImprovedCountries_B.index

In [ ]:
### cell 15 ###

Mobile_Stats.sort_values(by=['Change_Download'])

In [ ]:
### cell 16 ###

Broadband_Stats.sort_values(by=['Change_Download'])

In [ ]:
### cell 17 ###

# Use GPU‐friendly, vectorized first/last aggregations instead of Python lambdas
Mobile_stats = Mobile_df.groupby('Name').agg(
    first_Download=('Avg. Avg D Kbps', 'first'),
    last_Download =('Avg. Avg D Kbps', 'last'),
    first_Upload  =('Avg. Avg U Kbps', 'first'),
    last_Upload   =('Avg. Avg U Kbps', 'last'),
    first_Latency =('Avg Lat Ms',        'first'),
    last_Latency  =('Avg Lat Ms',        'last'),
)
Mobile_Stats_relative = Mobile_stats.assign(
    Change_Download=(Mobile_stats['last_Download']  - Mobile_stats['first_Download']) / Mobile_stats['first_Download'],
    Change_Upload  =(Mobile_stats['last_Upload']    - Mobile_stats['first_Upload'])  / Mobile_stats['first_Upload'],
    Change_Latency =(Mobile_stats['last_Latency']   - Mobile_stats['first_Latency']) / Mobile_stats['first_Latency'],
)[['Change_Download','Change_Upload','Change_Latency']]

Broadband_stats = Broadband_df.groupby('Name').agg(
    first_Download=('Avg. Avg D Kbps', 'first'),
    last_Download =('Avg. Avg D Kbps', 'last'),
    first_Upload  =('Avg. Avg U Kbps', 'first'),
    last_Upload   =('Avg. Avg U Kbps', 'last'),
    first_Latency =('Avg Lat Ms',        'first'),
    last_Latency  =('Avg Lat Ms',        'last'),
)
Broadband_Stats_relative = Broadband_stats.assign(
    Change_Download=(Broadband_stats['last_Download']  - Broadband_stats['first_Download']) / Broadband_stats['first_Download'],
    Change_Upload  =(Broadband_stats['last_Upload']    - Broadband_stats['first_Upload'])  / Broadband_stats['first_Upload'],
    Change_Latency =(Broadband_stats['last_Latency']   - Broadband_stats['first_Latency']) / Broadband_stats['first_Latency'],
)[['Change_Download','Change_Upload','Change_Latency']]

In [ ]:
### cell 18 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 2)]

In [ ]:
### cell 19 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 1) & (Broadband_Stats_relative['Change_Download'] < 2)]

In [ ]:
### cell 20 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 0.5) & (Broadband_Stats_relative['Change_Download'] < 1)]

In [ ]:
### cell 21 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 0.2) & (Broadband_Stats_relative['Change_Download'] < 0.5)]

In [ ]:
### cell 22 ###

# Use Series.between to fuse the two comparisons into one GPU call
ImprovedCountries_B = Broadband_Stats_relative[
    Broadband_Stats_relative['Change_Download'].between(0, 0.2, inclusive='left')
]

In [ ]:
### cell 23 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] < 0)]

In [ ]:
### cell 24 ###

Broadband_Stats_relative.sort_values(by=['Change_Download'])

In [ ]:
### cell 25 ###

Mobile_Stats_relative.sort_values(by=['Change_Download'])